In [28]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np

In [29]:
#=========
# Setup
#=========
df = pd.DataFrame({
    "region": ["East" , "East" , "West" , "West" , "North" , "East" , "West" , "North"] ,
    "product": ["Cable" , "Wire" , "Cable" , "Wire" , "Cable" , "Cable" , "Wire" , "Wire"] ,
    "month": pd.to_datetime(["2025-11" , "2025-11" , "2025-11" , "2025-12" , "2025-12" , "2025-12" , "2025-12" , "2025-11"]) ,
    "sales": [120 , 80 , 90 , 110 , 70 , 150 , 60 , 95] ,
    "returns": [5 , 2 , 3 , 6 , 1 , 4 , 7 , 2] ,
})
df["return_rate"] = df["returns"] / df["sales"]
df

,region,product,month,sales,returns,return_rate
0,East,Cable,2025-11-01,120,5,0.041667
1,East,Wire,2025-11-01,80,2,0.025000
2,West,Cable,2025-11-01,90,3,0.033333
3,West,Wire,2025-12-01,110,6,0.054545
4,North,Cable,2025-12-01,70,1,0.014286
5,East,Cable,2025-12-01,150,4,0.026667
6,West,Wire,2025-12-01,60,7,0.116667
7,North,Wire,2025-11-01,95,2,0.021053


In [30]:
# ============================================================
# Case 1) Boolean mask + .loc
# ============================================================
mask = df["region"].isin(["East" , "West"]) & (df["sales"] >= 100)
case1 = df.loc[mask , ["region" , "product" , "month" , "sales" , "returns"]]
case1

,region,product,month,sales,returns
0,East,Cable,2025-11-01,120,5
3,West,Wire,2025-12-01,110,6
5,East,Cable,2025-12-01,150,4


In [31]:
#===================================================
# Case 2) query() for readability (SQL-like WHERE)
#===================================================
case2 = df.query("region in ['East' , 'West'] and sales >= 100")
case2

,region,product,month,sales,returns,return_rate
0,East,Cable,2025-11-01,120,5,0.041667
3,West,Wire,2025-12-01,110,6,0.054545
5,East,Cable,2025-12-01,150,4,0.026667


In [32]:
#==============================================
# Case 3) query() with @ for local variables
#==============================================
regions = ["East" , "North"]
min_sales = 90
case3 = df.query("region in @regions and sales >= @min_sales")
case3

,region,product,month,sales,returns,return_rate
0,East,Cable,2025-11-01,120,5,0.041667
5,East,Cable,2025-12-01,150,4,0.026667
7,North,Wire,2025-11-01,95,2,0.021053


In [33]:
#=================================================
# Case 4) eval() for KPI math (derived columns)
#=================================================
case4 = df.copy()
case4.eval("net_sales = sales - returns" , inplace = True)
case4.eval("flag_high_returns = return_rate > 0.05" , inplace = True)
case4[["region" , "product" , "sales" , "returns" , "net_sales" , "flag_high_returns"]]

,region,product,sales,returns,net_sales,flag_high_returns
0,East,Cable,120,5,115,False
1,East,Wire,80,2,78,False
2,West,Cable,90,3,87,False
3,West,Wire,110,6,104,True
4,North,Cable,70,1,69,False
5,East,Cable,150,4,146,False
6,West,Wire,60,7,53,True
7,North,Wire,95,2,93,False


In [34]:
#===================================================================
# Case 5) eval/query engines: numexpr vs python
# Keep this as a pattern; performance gains show up on large data.
#===================================================================
expr = "(sales >= 90) & (returns <= 5) & (return_rate < 0.06)"
case5 = df.eval(expr , engine = "numexpr")
case5.head()

0     True
1    False
2     True
3    False
4    False
dtype: bool

In [35]:
#========================================================
# Case 6) SELECT + WHERE style: filter() then query()
#========================================================
kpi = df.filter(items = ["region" , "product" , "month" , "sales" , "returns"])
case6 = (
    kpi.query("month >= '2025-12-01' and product == 'Cable'")
        .sort_values(["region" , "sales"] ,  ascending = [True , False])
)
case6

,region,product,month,sales,returns
5,East,Cable,2025-12-01,150,4
4,North,Cable,2025-12-01,70,1
